# Notebook 02: Calidad, Limpieza y Preparación
## Proyecto Integrador — Minería de Datos I

**Objetivo:** Aplicar las transformaciones necesarias sobre el dataset, documentando para cada decisión: la evidencia que la motivó, la acción aplicada y el impacto observado. Preservar el dataset original sin modificaciones en `data/raw/` y guardar el resultado final en `data/processed/`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from datetime import datetime

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Rutas
RAW_PATH = '../data/raw/reporte_clinica_original.csv'
PROCESSED_PATH = '../data/processed/reporte_clinica_analisis.csv'
LOG_PATH = '../logs/pipeline_log.csv'

# Cargar dataset original
df = pd.read_csv(RAW_PATH)
print(f'Dataset original: {df.shape[0]} filas x {df.shape[1]} columnas')

# Inicializar log ETL
log_etl = []

def registrar_paso(paso, descripcion, df_antes, df_despues):
    """Registra cada transformación en el log ETL."""
    nulos_antes = df_antes.isnull().sum().sum()
    nulos_despues = df_despues.isnull().sum().sum()
    retencion = round(len(df_despues) / len(df_antes) * 100, 2) if len(df_antes) > 0 else 100
    log_etl.append({
        'Paso': paso,
        'Descripción': descripcion,
        'Filas': len(df_despues),
        'Nulos': nulos_despues,
        'Retención (%)': retencion
    })

## 1. Eliminación de columna índice (`Unnamed: 0`)

**Evidencia:** La columna `Unnamed: 0` contiene simplemente un índice numérico secuencial (0 a 1362) sin valor informativo. Es un artefacto de exportación.

**Acción:** Eliminar la columna.

**Impacto:** Se reduce de 8 a 7 columnas sin pérdida de información relevante.

In [ ]:
df_antes = df.copy()
df = df.drop(columns=['Unnamed: 0'])

registrar_paso(1, 'Eliminación de columna índice redundante (Unnamed: 0)', df_antes, df)
print(f'Columnas antes: {list(df_antes.columns)}')
print(f'Columnas después: {list(df.columns)}')
print(f'Reducción: {len(df_antes.columns)} → {len(df.columns)} columnas')

## 2. Estandarización de variables categóricas

### 2.1 Columna `sex`

**Evidencia:** Se detectaron 4 variantes (`female`, `male`, `Female`, `Male`) cuando solo deberían existir 2. La diferencia es únicamente de capitalización.

**Acción:** Convertir todo a minúsculas.

**Impacto:** Reducción de 4 a 2 categorías, eliminando ambigüedad.

In [ ]:
print('Antes:', sorted(df['sex'].dropna().unique()))
print('Frecuencias antes:')
print(df['sex'].value_counts())

df_antes = df.copy()
df['sex'] = df['sex'].str.lower()

print(f'\nDespués: {sorted(df["sex"].dropna().unique())}')
print('Frecuencias después:')
print(df['sex'].value_counts())

registrar_paso(2, 'Estandarización de sex: Female→female, Male→male', df_antes, df)

### 2.2 Columna `smoker`

**Evidencia:** Igual que `sex`: 4 variantes (`yes`, `no`, `Yes`, `No`) por diferencia de capitalización.

**Acción:** Convertir todo a minúsculas.

**Impacto:** Reducción de 4 a 2 categorías.

In [ ]:
print('Antes:', sorted(df['smoker'].dropna().unique()))
print('Frecuencias antes:')
print(df['smoker'].value_counts())

df_antes = df.copy()
df['smoker'] = df['smoker'].str.lower()

print(f'\nDespués: {sorted(df["smoker"].dropna().unique())}')
print('Frecuencias después:')
print(df['smoker'].value_counts())

registrar_paso(3, 'Estandarización de smoker: Yes→yes, No→no', df_antes, df)

### 2.3 Columna `region`

**Evidencia:** Se detectaron 8 variantes mezclando siglas (`SE`, `NE`, `SW`, `NW`) con nombres completos (`southeast`, `northeast`, `southwest`, `northwest`). Representan las mismas 4 regiones.

**Acción:** Convertir siglas a nombres completos en minúsculas.

**Impacto:** Reducción de 8 a 4 categorías.

In [ ]:
print('Antes:', sorted(df['region'].dropna().unique()))
print('Frecuencias antes:')
print(df['region'].value_counts())

df_antes = df.copy()

# Mapeo de siglas a nombres completos
region_map = {
    'SE': 'southeast',
    'NE': 'northeast',
    'SW': 'southwest',
    'NW': 'northwest'
}
df['region'] = df['region'].replace(region_map).str.lower()

print(f'\nDespués: {sorted(df["region"].dropna().unique())}')
print('Frecuencias después:')
print(df['region'].value_counts())

registrar_paso(4, 'Estandarización de region: siglas (SE,NE,SW,NW) → nombres completos', df_antes, df)

## 3. Tratamiento de outliers

### 3.1 Outliers en `bmi`

**Evidencia:** Se detectaron valores fisiológicamente imposibles:
- 14 registros con `bmi ≤ 0` (incluye el valor -5)
- 6 registros con `bmi ≥ 100` (incluye el valor 999)

Un IMC (BMI) negativo o de 999 no tiene sentido clínico.

**Acción:** Reemplazar estos valores por `NaN` para tratarlos junto con los demás nulos.

**Impacto:** Se invalidan 20 registros de BMI que serán imputados posteriormente.

In [ ]:
print('=== BMI ANTES DE LIMPIEZA ===')
print(f'bmi ≤ 0: {(df["bmi"] <= 0).sum()} registros')
print(f'bmi ≥ 100: {(df["bmi"] >= 100).sum()} registros')

# Mostrar los valores extremos
extremos_bmi = df[(df['bmi'] <= 0) | (df['bmi'] >= 100)]
print(f'\nRegistros con BMI extremo ({len(extremos_bmi)}):')
extremos_bmi[['age', 'sex', 'bmi', 'children', 'charges']]

In [ ]:
df_antes = df.copy()
n_antes = df['bmi'].isnull().sum()

# Reemplazar outliers por NaN
df.loc[df['bmi'] <= 0, 'bmi'] = np.nan
df.loc[df['bmi'] >= 100, 'bmi'] = np.nan

n_despues = df['bmi'].isnull().sum()
print(f'Nulos en bmi antes: {n_antes}')
print(f'Nulos en bmi después: {n_despues}')
print(f'Nuevos nulos agregados: {n_despues - n_antes}')

registrar_paso(5, f'Outliers BMI convertidos a NaN: {n_despues - n_antes} registros (≤0 o ≥100)', df_antes, df)

### 3.2 Outliers en `children`

**Evidencia:** Se detectaron valores fuera de rango razonable:
- 2 registros con `children = -1` (imposible)
- 8 registros con `children > 10` (15 y 99)

**Acción:** Para valores negativos: reemplazar por `NaN`. Para valores > 10: reemplazar por `NaN` al ser biológicamente improbables.

**Impacto:** Se invalidan 10 registros de `children`.

In [ ]:
print('=== CHILDREN ANTES DE LIMPIEZA ===')
print(f'children < 0: {(df["children"] < 0).sum()} registros')
print(f'children > 10: {(df["children"] > 10).sum()} registros')

extremos_ch = df[(df['children'] < 0) | (df['children'] > 10)]
print(f'\nRegistros con children extremo ({len(extremos_ch)}):')
extremos_ch[['age', 'sex', 'bmi', 'children', 'charges']]

In [ ]:
df_antes = df.copy()

# Reemplazar outliers por NaN
df.loc[df['children'] < 0, 'children'] = np.nan
df.loc[df['children'] > 10, 'children'] = np.nan

print(f'Distribución de children después de limpiar outliers:')
print(df['children'].value_counts().sort_index())

registrar_paso(6, 'Outliers children convertidos a NaN: 10 registros (<0 o >10)', df_antes, df)

## 4. Imputación de valores nulos

### Estado actual de nulos tras limpieza de outliers

In [ ]:
print('=== NULOS ACTUALES ===')
nulos_actuales = df.isnull().sum()
print(nulos_actuales[nulos_actuales > 0])
print(f'\nTotal: {df.isnull().sum().sum()} nulos')
print(f'Filas con al menos un nulo: {df.isnull().any(axis=1).sum()}')

### 4.1 Imputación de `age`

**Evidencia:** 102 nulos (7.48%). La edad tiene distribución relativamente simétrica. La mediana es más robusta frente a outliers que la media.

**Acción:** Imputar con la **mediana**.

**Impacto:** Se completa la columna `age` sin valores faltantes.

In [ ]:
print(f'Mediana de age: {df["age"].median():.1f}')
print(f'Media de age: {df["age"].mean():.1f}')

df_antes = df.copy()
n_nulos_age = df['age'].isnull().sum()
mediana_age = df['age'].median()
df['age'] = df['age'].fillna(mediana_age)

print(f'\nNulos imputados en age: {n_nulos_age}')
print(f'Nulos restantes en age: {df["age"].isnull().sum()}')

registrar_paso(7, f'Imputación de age: {n_nulos_age} nulos → mediana ({mediana_age:.1f})', df_antes, df)

### 4.2 Imputación de `bmi`

**Evidencia:** 50 nulos originales + 20 nuevos por outliers = **70 nulos** (5.14%). La distribución del BMI está sesgada. La mediana es más apropiada.

**Acción:** Imputar con la **mediana**.

**Impacto:** Se completa la columna `bmi` sin valores faltantes.

In [ ]:
print(f'Mediana de bmi (sin outliers): {df["bmi"].median():.2f}')
print(f'Media de bmi (sin outliers): {df["bmi"].mean():.2f}')

df_antes = df.copy()
n_nulos_bmi = df['bmi'].isnull().sum()
mediana_bmi = df['bmi'].median()
df['bmi'] = df['bmi'].fillna(mediana_bmi)

print(f'\nNulos imputados en bmi: {n_nulos_bmi}')
print(f'Nulos restantes en bmi: {df["bmi"].isnull().sum()}')

registrar_paso(8, f'Imputación de bmi: {n_nulos_bmi} nulos → mediana ({mediana_bmi:.2f})', df_antes, df)

### 4.3 Imputación de `children`

**Evidencia:** 10 nulos generados por outliers. La variable es de conteo con distribución asimétrica.

**Acción:** Imputar con la **mediana**. Dado que el 57% tiene 0 o 1 hijo, la mediana (1) es representativa.

**Impacto:** Se completa la columna `children` sin valores faltantes.

In [ ]:
print(f'Mediana de children: {df["children"].median():.0f}')
print(f'Moda de children: {df["children"].mode()[0]}')
print(f'\nDistribución de children:')
print(df['children'].value_counts().sort_index())

df_antes = df.copy()
n_nulos_ch = df['children'].isnull().sum()
mediana_ch = df['children'].median()
df['children'] = df['children'].fillna(mediana_ch)

print(f'\nNulos imputados en children: {n_nulos_ch}')
print(f'Nulos restantes: {df["children"].isnull().sum()}')

registrar_paso(9, f'Imputación de children: {n_nulos_ch} nulos → mediana ({mediana_ch:.0f})', df_antes, df)

## 5. Verificación final

In [ ]:
print('=== VERIFICACIÓN FINAL ===')
print(f'Filas: {df.shape[0]}')
print(f'Columnas: {df.shape[1]}')
print(f'Nulos totales: {df.isnull().sum().sum()}')
print(f'Duplicados: {df.duplicated().sum()}')
print()
print('Tipos de datos:')
print(df.dtypes)
print()
print('Estadísticas descriptivas:')
df.describe()

In [ ]:
# Verificar valores categóricos
print('=== VALORES CATEGÓRICOS FINALES ===')
print(f'sex: {sorted(df["sex"].unique())}')
print(f'smoker: {sorted(df["smoker"].unique())}')
print(f'region: {sorted(df["region"].unique())}')
print(f'children: {sorted(df["children"].unique())}')

## 6. Guardar dataset procesado

In [ ]:
# Guardar dataset limpio
df.to_csv(PROCESSED_PATH, index=False)
print(f'✅ Dataset procesado guardado en: {PROCESSED_PATH}')
print(f'   Filas: {df.shape[0]}, Columnas: {df.shape[1]}, Nulos: {df.isnull().sum().sum()}')

# Verificar que el original sigue intacto
df_original = pd.read_csv(RAW_PATH)
print(f'\n✅ Dataset original intacto en: {RAW_PATH}')
print(f'   Filas: {df_original.shape[0]}, Columnas: {df_original.shape[1]}, Nulos: {df_original.isnull().sum().sum()}')

## 7. Guardar Log ETL

In [ ]:
# Crear y guardar log ETL
log_df = pd.DataFrame(log_etl)
log_df.to_csv(LOG_PATH, index=False)

print('=== LOG ETL (pipeline_log.csv) ===')
print(log_df.to_string(index=False))
print(f'\n✅ Log guardado en: {LOG_PATH}')

## 8. Resumen de transformaciones

| Paso | Transformación | Impacto |
|---|---|---|
| 1 | Eliminar columna `Unnamed: 0` | 8 → 7 columnas |
| 2 | Estandarizar `sex` a minúsculas | 4 → 2 categorías |
| 3 | Estandarizar `smoker` a minúsculas | 4 → 2 categorías |
| 4 | Estandarizar `region`: siglas → nombres | 8 → 4 categorías |
| 5 | Convertir outliers `bmi` a NaN | +20 nulos |
| 6 | Convertir outliers `children` a NaN | +10 nulos |
| 7 | Imputar `age` con mediana | 102 nulos → 0 |
| 8 | Imputar `bmi` con mediana | 70 nulos → 0 |
| 9 | Imputar `children` con mediana | 10 nulos → 0 |

### Resultado final
- **1,363 filas × 7 columnas**
- **0 valores nulos**
- **0 duplicados**
- Categóricas estandarizadas
- Dataset original preservado en `data/raw/`